# EXtaire les tokens

In [1]:
import requests
import json

In [4]:
prompt32k = "lol"

In [3]:
# Configuration
URL = "https://unpalpablely-vibronic-leonore.ngrok-free.dev/api/v1/chat/completions"
API_KEY = "sk-88cbf160a9574be489dc529a29e5aab7"
MODEL_NAME = "qwen/qwen3-30b-a3b-2507"


def envoyer_requete(prompt):
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {API_KEY}",
    }

    data = {
        "model": MODEL_NAME,
        "messages": [{"role": "user", "content": prompt}],
        # Désactive le streaming pour avoir une réponse d'un bloc
        "stream": False,
        "max_tokens": 20,
    }

    try:
        response = requests.post(URL, headers=headers, data=json.dumps(data))
        response.raise_for_status()  # Vérifie si la requête a réussi (code 200)

        resultat = response.json()
        return resultat

    except requests.exceptions.RequestException as e:
        return f"Erreur de connexion : {e}"
    except KeyError:
        return "Erreur : Format de réponse inattendu."


# Test


if __name__ == "__main__":
    reponse = envoyer_requete(prompt32k)
    print(reponse)
    # message = reponse["choices"][0]["message"]["content"]
    # print(message)

{'id': 'chatcmpl-35bhjylh5toxsy2102bx2', 'object': 'chat.completion', 'created': 1774861188, 'model': 'qwen/qwen3-30b-a3b-2507', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': '**L’Âge d’or de la piraterie dans les Caraïbes :', 'tool_calls': []}, 'logprobs': None, 'finish_reason': 'length'}], 'usage': {'prompt_tokens': 47302, 'completion_tokens': 19, 'total_tokens': 47321}, 'stats': {}, 'system_fingerprint': 'qwen/qwen3-30b-a3b-2507'}


In [31]:
reponse

{'id': 'chatcmpl-9m91e867k1p76l4pw1211t',
 'object': 'chat.completion',
 'created': 1774601603,
 'model': 'qwen/qwen3-30b-a3b-2507',
 'choices': [{'index': 0,
   'message': {'role': 'assistant',
    'content': '**L’Âge d’or de la piraterie dans les Caraïbes : Un récit cinématographique, historique et psychologique du Queen Anne’s Revenge**\n\n---\n\n### **I. Le vaisseau : un rêve d’émancipation flottant sur les flots azurs**\n\nLe *Queen Anne’s Revenge*, capturé en 1718 par Edward Teach — plus connu sous le nom de **Blackbeard** — n’était pas un simple navire de guerre. C’était une **machine de liberté**, un vaisseau de 100 pieds de long, dont le **beaupré** d’acier s’effilait comme une épée de mer, les **haubans** tendus comme des nerfs de l’océan, et le **gaillard d’arrière** couvert de croix de Saint-André et de symboles de souveraineté. Sa coque de chêne massif, fendue par des boulets de canon, portait encore les marques des batailles livrées contre la Royal Navy — une guerre de **

In [27]:
print('Input tokens:', reponse['usage']['prompt_tokens'])
print('Output tokens:', reponse['usage']['completion_tokens'])
print('Total tokens:', reponse['usage']['total_tokens'])

TypeError: string indices must be integers, not 'str'

In [ ]:
def envoyer_requete_avec_usage(prompt):
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {API_KEY}",
    }
    data = {
        "model": MODEL_NAME,
        "messages": [{"role": "user", "content": prompt}],
        "stream": True,
        "stream_options": {"include_usage": True},  # <--- INDISPENSABLE pour voir l'usage en stream
        "max_tokens" : 20
    }

    try:
        response = requests.post(URL, headers=headers, json=data, stream=True)
        response.raise_for_status()

        texte_complet = ""
        stats_usage = None

        for line in response.iter_lines():
            if not line:
                continue
                
            line_text = line.decode('utf-8').replace('data: ', '').strip()
            if line_text == "[DONE]":
                break
            
            try:
                chunk = json.loads(line_text)
                
                # On récupère le texte (delta)
                if "choices" in chunk and len(chunk["choices"]) > 0:
                    content = chunk["choices"][0].get("delta", {}).get("content", "")
                    if content:
                        print(content, end="", flush=True)
                        texte_complet += content
                
                # On récupère l'usage
                if "usage" in chunk and chunk["usage"] is not None:
                    stats_usage = chunk["usage"]

            except json.JSONDecodeError:
                continue
        
        print("\n" + "-"*30)
        if stats_usage:
            print(f"Tokens Entrée (Prompt) : {stats_usage.get('prompt_tokens')}")
            print(f"Tokens Sortie (Réponse) : {stats_usage.get('completion_tokens')}")
            print(f"Total Tokens : {stats_usage.get('total_tokens')}")
        else:
            print("Usage non disponible (le serveur n'a pas renvoyé les stats).")
            
        return texte_complet, stats_usage

    except Exception as e:
        return f"Erreur : {e}", None

if __name__ == "__main__":
    reponse, stats = envoyer_requete_avec_usage(prompt32k)
    #print(reponse)
    print(stats)
    #message = reponse["choices"][0]["message"]["content"]
    #print(message)

**L’Âge d’or de la piraterie dans les Caraïbes : une épopée maritime de liberté, de terreur et de démocratie brute**

---

### **I. Le *Queen Anne’s Revenge* : navire de rêve, navire de révolte**

Le *Queen Anne’s Revenge*, galion de 70 canons, capturé en 1718 par Edward Teach — mieux connu sous le nom de **Blackbeard** — n’était pas un vaisseau de guerre ordinaire. Il était un **symbole**. Un vaisseau d’abordage, dépourvu de la rigidité des flottes royales, conçu pour la vitesse, la manœuvrabilité et l’effroi. Son chêne, fendu par les vents du large, portait encore les marques du combat : des déchirures laissées par des boulets espagnols, des taches de sang séché dans les joints du pont, et le frottement des haubans qui gémissaient comme des âmes en peine.

Mais ce qui le distinguait, c’était sa **structure sociale**. À bord, pas de capitaine tyrannique, pas de hiérarchie rigide. Le **code de piraterie** — véritable constitution flottante — était appliqué avec une rigueur quasi sacrée